In [1]:
# =====================================================================
# STAGE 1: Environment Cleanup and Initialization (Pre-LLM Loading)
# =====================================================================

# Optional: Dependency conflict resolution. 
# Uncomment the lines below to flush old packages and restart the session if necessary.
# !rm -rf /usr/local/lib/python3.12/dist-packages/transformers*
# !rm -rf /usr/local/lib/python3.12/dist-packages/huggingface_hub*
# !pip install --no-cache-dir -U gradio huggingface_hub transformers accelerate bitsandbytes

import sys
import gc

# ---------------------------------------------------------
# [1] Compatibility Hotfixes for Hugging Face & Transformers
# ---------------------------------------------------------
try:
    import huggingface_hub.utils._auth as _auth
    if not hasattr(_auth, '_write_secret'): _auth._write_secret = lambda *args, **kwargs: None
except: pass

try:
    import huggingface_hub.hf_api as hf_api
    if not hasattr(hf_api, 'KernelInfo'): hf_api.KernelInfo = type('DummyKernelInfo', (), {})
except: pass

try:
    import transformers.utils.generic as generic
    if not hasattr(generic, 'add_model_info_to_auto_map'):
        generic.add_model_info_to_auto_map = lambda *args, **kwargs: None
except: pass

# ---------------------------------------------------------
# [2] Core Library Imports
# ---------------------------------------------------------
import os
import glob
import re
import json
import random
import torch
import gradio as gr
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# ---------------------------------------------------------
# [3] VRAM Cleanup and Hardware Detection
# ---------------------------------------------------------
gc.collect()
torch.cuda.empty_cache()

device = "cuda" if torch.cuda.is_available() else "cpu"

print("==================================================")
print("ENVIRONMENT STATUS: Initialization successful.")
print(f"DEVICE STATUS: Running on {device.upper()}")
print("==================================================")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.1/20.1 MB 107.0 MB/s eta 0:00:00 0:00:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 292.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 671.5/671.5 kB 335.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 105.7 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 311.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 91.0 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 346.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 104.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.3/73.3 kB 301.4 MB/s eta 0:00:00
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.3.0
    Uninstalling hf-xet-1.3.0:
      Successfully uninstalled hf-xet-1.3.0
  Attempting uninstall: click
    Found existing installation: click 8.3.1
    Uninstalling

In [2]:
# =====================================================================
# STAGE 2: Model and Tokenizer Initialization
# =====================================================================

# Define the model path (adjust if deploying outside Kaggle)
MODEL_ID = "/kaggle/input/models/google/gemma-4/transformers/gemma-4-31b-it/1"

print("==================================================")
print("Initializing model loading sequence...")
print(f"Target Model: {MODEL_ID.split('/')[-2] if '/' in MODEL_ID else MODEL_ID}")
print("==================================================")

# ---------------------------------------------------------
# [1] Load Tokenizer
# ---------------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# ---------------------------------------------------------
# [2] Configure 4-bit Quantization
# ---------------------------------------------------------
# Utilizing Normal Float 4 (NF4) with double quantization for VRAM efficiency
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

# ---------------------------------------------------------
# [3] Load Base Model
# ---------------------------------------------------------
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    quantization_config=quantization_config,
    low_cpu_mem_usage=True
)

# Tie weights to ensure embedding and lm_head share the same parameters
model.tie_weights()

print("Model and tokenizer loaded successfully with 4-bit quantization.")

SYSTEM INITIALIZATION: LOADING INTELLECTUAL MATRIX


Loading weights:   0%|          | 0/1188 [00:00<?, ?it/s]

✅ SUCCESS: GEMMA-4 CORE INTRODUCED SUCCESSFULLY WITH 4-BIT QUANTIZATION.


In [10]:
# =====================================================================
# STAGE 3: Cognitive Tensor Engine (Dual-Track Graph Mapping)
# =====================================================================

class DualTrackCognitiveEngine:
    def __init__(self):
        # Resolve database path dynamically (supports Kaggle and local environments)
        paths = glob.glob("/kaggle/input/**/nexus_db.json", recursive=True)
        if paths: 
            self.db_path = paths[0]
        elif os.path.exists("/kaggle/working/nexus_db.json"): 
            self.db_path = "/kaggle/working/nexus_db.json"
        else: 
            raise FileNotFoundError("Initialization Failed: nexus_db.json not found in the environment.")
            
        self.node_registry = []
        self.mastered_nodes = {}
        self.k, self.m, self.n = 0, 0, 0
        self.last_active_node_id, self.last_score = None, 1.0 
        
        self._initialize_from_db()

    def _initialize_from_db(self):
        with open(self.db_path, "r", encoding="utf-8") as f: 
            db_data = json.load(f)
            
        # Register base knowledge nodes
        for base in db_data.get("base_nodes", []):
            self.node_registry.append({
                "is_empty": True, 
                "id": f"BASE_{len(self.node_registry)}", 
                "name": base["name"], 
                "type": base.get("type", "thought"), 
                "difficulty": 0.5
            })
        self.k = len(self.node_registry)
        
        # Register content nodes
        for content in db_data.get("content_nodes", []):
            self.node_registry.append({
                "is_empty": False, 
                "id": content["id"], 
                "title": content["title"], 
                "type": content["type"], 
                "rich_text": content.get("rich_text", ""), 
                "linked_bases": content.get("linked_bases", []), 
                "difficulty": content.get("difficulty", 0.5)
            })
        self.n = len(self.node_registry)
        
        # Initialize 3D cognitive tensor
        self.tensor = torch.zeros((self.n, self.n, 3), dtype=torch.float32)
        
        # Map node properties and dependencies
        for i in range(self.k, self.n):
            self.tensor[i, i, 1] = self.node_registry[i]["difficulty"]
            for b_name in self.node_registry[i]["linked_bases"]:
                b_idx = next((j for j in range(self.k) if self.node_registry[j]["name"] == b_name), None)
                if b_idx is not None:
                    self.tensor[b_idx, i, 0] = 1.0
                    self.tensor[i, b_idx, 0] = 1.0
                    
        for i in range(self.n): 
            self.tensor[i, i, 2] = 0.1

    def backpropagate_score(self, content_id, score, alpha=0.6):
        c_idx = next((i for i in range(self.n) if self.node_registry[i]["id"] == content_id), None)
        if c_idx is None or c_idx < self.k: 
            return
            
        self.last_active_node_id, self.last_score = content_id, score
        self.tensor[c_idx, c_idx, 2] = score
        
        # Update linked base node mastery via exponential moving average
        for b_idx in range(self.k):
            if self.tensor[b_idx, c_idx, 0] > 0:
                current_mastery = self.tensor[b_idx, b_idx, 2].item()
                self.tensor[b_idx, b_idx, 2] = (1 - alpha) * current_mastery + alpha * score
        
        # Mastery Vault Interceptor: Archive node if score meets the 90% threshold
        if score >= 0.9 and content_id not in self.mastered_nodes:
            self.mastered_nodes[content_id] = {
                "title": self.node_registry[c_idx]["title"],
                "rich_text": self.node_registry[c_idx]["rich_text"],
                "linked_bases": self.node_registry[c_idx]["linked_bases"],
                "notes": ""
            }

    def recommend_next_content(self):
        base_scores = torch.diagonal(self.tensor[:self.k, :self.k, 2])
        recs, last_vec = [], None
        
        if self.last_active_node_id and self.last_score < 0.6:
            lf_idx = next((i for i in range(self.n) if self.node_registry[i]["id"] == self.last_active_node_id), None)
            if lf_idx is not None:
                last_vec = self.tensor[:self.k, lf_idx, 0]
        
        for i in range(self.k, self.n):
            if self.node_registry[i]["id"] == self.last_active_node_id: 
                continue 
                
            deps = self.tensor[:self.k, i, 0]
            if torch.sum(deps).item() == 0: 
                continue
            
            readiness = torch.sum(base_scores * deps) / torch.sum(deps).item()
            weight = 1.0 - torch.abs(readiness - self.tensor[i, i, 1].item()) * 2.0
            
            if last_vec is not None and torch.sum(last_vec * deps).item() > 0: 
                weight += 0.3 * torch.sum(last_vec * deps).item()
            if self.tensor[i, i, 2].item() > 0.6: 
                weight -= (self.tensor[i, i, 2].item() - 0.6) * 1.5 
                
            recs.append({"index": i, "weight": weight})
            
        if not recs: 
            return {"status": "empty"}
            
        recs.sort(key=lambda x: x["weight"], reverse=True)
        node = self.node_registry[recs[0]["index"]]
        
        return {
            "status": "success", 
            "recommended_id": node["id"], 
            "title": node["title"], 
            "rich_text": node["rich_text"], 
            "linked_bases": node["linked_bases"]
        }

    def diagnose_weak_bases(self, content_id):
        c_idx = next((i for i in range(self.n) if self.node_registry[i]["id"] == content_id), None)
        if c_idx is None:
            return []
    
        weak_nodes = []
        for b_idx in range(self.k):
            if self.tensor[b_idx, c_idx, 0] > 0:
                mastery = self.tensor[b_idx, b_idx, 2].item()
                weak_nodes.append({
                    "name": self.node_registry[b_idx]["name"],
                    "mastery": mastery
                })
    
        weak_nodes.sort(key=lambda x: x["mastery"])
        return weak_nodes

    def update_base_node_mastery(self, mastered_nodes, weak_nodes, beta=0.15):
        for node_name in mastered_nodes:
            idx = next((i for i in range(self.k) if self.node_registry[i]["name"] == node_name), None)
            if idx is not None:
                self.tensor[idx, idx, 2] = min(1.0, self.tensor[idx, idx, 2].item() + beta)
    
        for node_name in weak_nodes:
            idx = next((i for i in range(self.k) if self.node_registry[i]["name"] == node_name), None)
            if idx is not None:
                self.tensor[idx, idx, 2] = max(0.05, self.tensor[idx, idx, 2].item() - beta * 0.5)

    def export_graph_state(self, content_id):
        weak_nodes = self.diagnose_weak_bases(content_id)
        return "\n".join([f"{x['name']} : {x['mastery']:.2f}" for x in weak_nodes])

    def update_node_evidence(self, node_name, evidence, delta):
        b_idx = next((i for i in range(self.k) if self.node_registry[i]["name"] == node_name), None)
        if b_idx is None:
            return
    
        old_score = self.tensor[b_idx, b_idx, 2].item()
        new_score = 0.8 * old_score + 0.2 * evidence + delta
        new_score = max(0.0, min(1.0, new_score))
        
        self.tensor[b_idx, b_idx, 2] = new_score
        return new_score

In [40]:
# =====================================================================
# STAGE 4: Text Processing Utilities & LLM Inference Handlers
# =====================================================================

def split_q_and_a(rich_text):
    """
    Splits a single rich text block into Question and Answer components 
    based on predefined LaTeX delimiters to prevent spoilers.
    """
    parts = re.split(r'(\\textbf\{[解析证明]+[：\:]?\})', rich_text, maxsplit=1)
    if len(parts) >= 3: 
        return parts[0], parts[1] + parts[2]
    return rich_text, "*No official solution provided in the database.*"


def format_math_for_gradio(text):
    """
    Sanitizes and converts LaTeX formatting into a Gradio-compatible Markdown format.
    Specifically handles the escaping of set braces { } to prevent Markdown parser conflicts.
    """
    if not text:
        return ""

    # Remove leading whitespaces
    text = "\n".join([line.lstrip() for line in text.split("\n")])

    # Core Regex Fix: Escapes standalone set braces {1, 2} without affecting \textbf{} or x^{2}
    text = re.sub(
        r"(?<![a-zA-Z0-9_\^\\\}])\{((?:[^{}]|\{[^{}]*\})*)\}",
        r"\\{\1\\}",
        text
    )

    # Standardize math block delimiters
    text = text.replace("\\[", "\n$$\n").replace("\\]", "\n$$\n")
    text = text.replace("\\(", "$").replace("\\)", "$")

    # Convert basic LaTeX styling to standard Markdown
    text = re.sub(r"\\textbf\{(.*?)\}", r"**\1**", text)
    text = re.sub(r"\\textit\{(.*?)\}", r"*\1*", text)
    
    # Clean itemized lists
    text = text.replace("\\begin{itemize}", "").replace("\\end{itemize}", "").replace("\\item", "• ")

    return text


# Initial baseline questions for cold-start navigation
BASELINE_QUESTIONS = [
    "Given the function \\[ f(x) = x^3 - 3x^2 + ax + 1 \\] has a local minimum at \\( x=1 \\), find the value of the real number \\( a \\) and determine the monotonic intervals of the function.",
    "Given planar vectors \\( \\vec{a} = (1, 2) \\) and \\( \\vec{b} = (m, -1) \\). If \\( (\\vec{a} + \\vec{b}) \\perp \\vec{a} \\), find the value of the real number \\( m \\).",
    "Let \\( x > 0, y > 0 \\), and \\( x + 2y = 4 \\). Find the minimum value of the algebraic expression \\[ \\frac{1}{x} + \\frac{2}{y} \\]."
]


def extract_json(text):
    """
    Robust JSON extractor using stack-based depth tracking.
    Handles extra text injected by the LLM before or after the JSON payload.
    """
    if not text:
        return None

    text = text.replace("```json", "").replace("```", "")
    start = text.find("{")
    
    if start == -1:
        return None

    depth = 0
    for i in range(start, len(text)):
        if text[i] == "{":
            depth += 1
        elif text[i] == "}":
            depth -= 1
            if depth == 0:
                return text[start:i+1]
                
    return None


def gemma_generate_json(system_instruction):
    """
    Inference handler forcing structured JSON output from Gemma.
    """
    try:
        chat = [{"role": "user", "content": system_instruction}]
        prompt = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors="pt").to(device)

        # Use torch.no_grad() during inference to save memory
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=300,
                temperature=0.1
            )

        response_text = tokenizer.decode(
            outputs[0][inputs["input_ids"].shape[1]:],
            skip_special_tokens=True
        )

        json_str = extract_json(response_text)
        
        if json_str is None:
            return json.dumps({
                "score": 30,
                "feedback": "Model generated invalid structural output. Failed to parse JSON.",
                "status": "INCORRECT"
            })

        return json_str

    except Exception as e:
        return json.dumps({
            "score": 30,
            "feedback": f"System Engine Error: {str(e)}",
            "status": "INCORRECT"
        })


def gemma_generate_text(system_instruction):
    """
    Inference handler for generating standard, unstructured text feedback.
    """
    try:
        chat = [{"role": "user", "content": system_instruction}]
        prompt = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs, 
                max_new_tokens=400, 
                temperature=0.3
            )
            
        return tokenizer.decode(
            outputs[0][inputs["input_ids"].shape[1]:], 
            skip_special_tokens=True
        ).strip()
        
    except Exception as e: 
        return f"Tutor Engine encountered a runtime error: {e}"

In [52]:
# =====================================================================
# STAGE 5: Interactive Chat & Tutor Flow Control
# =====================================================================

def chat_with_tutor(user_msg, chat_history, session_manager):
    """
    Handles free-form conversational queries with the AI tutor.
    Injects current problem context to provide heuristic hints without revealing the final answer.
    """
    if not user_msg:
        return "", chat_history

    q_text = getattr(session_manager, "_current_q_text", "Unknown")
    chat_history.append({"role": "user", "content": user_msg})

    prompt = f"""
You are a top-tier math coach.

Current Problem:
{q_text}

Student Question:
{user_msg}

Give a heuristic hint. Do NOT reveal final answer.
"""

    reply = gemma_generate_text(prompt)
    chat_history.append({
        "role": "assistant",
        "content": format_math_for_gradio(reply)
    })

    return "", chat_history


def request_ai_hint(chat_history, session_manager):
    """
    Graph-aware hint generator. 
    Retrieves the student's weak knowledge nodes from the cognitive tensor to generate scaffolded micro-problems.
    """
    q_text = getattr(session_manager, "_current_q_text", "Unknown")
    tools = getattr(session_manager, "_current_tools", [])
    
    weak_nodes = session_manager.diagnose_weak_bases(session_manager._current_quest_id)
    graph_state = getattr(session_manager, "_graph_state", "N/A")

    weak_text = "\n".join([f"- {x['name']} (mastery={x['mastery']:.2f})" for x in weak_nodes])

    # UI Request Indicator
    chat_history.append({
        "role": "user",
        "content": "💡 Request hint"
    })

    prompt = f"""
You are an elite mathematics tutor.

Current Problem:
{q_text}

Official Knowledge Tags:
{tools}

Weak Concepts:
{weak_text}

Graph State:
{graph_state}

Task:
1. Identify the core bottleneck based on the weak concepts.
2. Generate 3 scaffolded micro-problems.
3. Do NOT reveal the final answer to the current problem.

Format:
[Core Bottleneck]
...

[Micro Problem 1]
...

[Micro Problem 2]
...

[Micro Problem 3]
...
"""

    reply = gemma_generate_text(prompt)
    chat_history.append({
        "role": "assistant",
        "content": format_math_for_gradio(reply)
    })

    return chat_history


def submit_answer_and_evaluate(user_ans, chat_history, session_manager):
    """
    Evaluates the student's submitted solution, extracts partial/full credit, 
    and triggers backpropagation in the cognitive tensor to update node mastery.
    """
    if not user_ans:
        return gr.update(), chat_history, gr.update(), gr.update(), gr.update(), session_manager

    q_id = getattr(session_manager, "_current_quest_id", None)
    q_text = getattr(session_manager, "_current_q_text", "Unknown")
    a_text = getattr(session_manager, "_current_a_text", "None")

    # UI Submission Indicator
    chat_history.append({
        "role": "user",
        "content": f"📝 Submitted Solution:\n{user_ans}"
    })

    # Enhanced Prompt: Enforcing generous partial credit for correct methodological approaches
    prompt = f"""Evaluate the student's math solution.
Problem: {q_text}
Student Answer: {user_ans}

Grading Rubric (Strictly follow this):
- CORRECT (85-100): Flawless logic and correct final analytical expression.
- PARTIAL (50-84): The student identified the correct structural approach, key insight, or substitution method (e.g., substituting x with 1/x). You MUST generously reward correct thinking and direction, even if the final calculation is incomplete or missing. Give them a high partial score (e.g., 60-80) for finding the right path!
- INCORRECT (0-49): Completely wrong direction or fundamental misunderstanding.

Provide encouraging feedback. Return JSON only format exactly like this:
{{"score": 0-100, "feedback": "...", "status": "CORRECT|PARTIAL|INCORRECT"}}"""

    try:
        data = json.loads(gemma_generate_json(prompt))
        score = float(data.get("score", 30)) / 100.0
        feedback = data.get("feedback", "")
        status = data.get("status", "INCORRECT")
    except Exception as e:
        score = 0.35
        feedback = f"Parsing error during evaluation: {str(e)}"
        status = "INCORRECT"

    # Propagate the evaluation score back into the knowledge graph
    session_manager.backpropagate_score(q_id, score)

    prefix = "🟢 CORRECT" if status == "CORRECT" else (
        "🟡 PARTIAL" if status == "PARTIAL" else "🔴 INCORRECT"
    )

    chat_history.append({
        "role": "assistant",
        "content": f"{prefix} ({int(score*100)})\n\n{feedback}"
    })

    # UI State Updates (hide input, show answer block)
    return (
        gr.update(visible=False), 
        chat_history, 
        gr.update(visible=True), 
        format_math_for_gradio(a_text), 
        gr.update(visible=True), 
        session_manager
    )

In [42]:
# =====================================================================
# Stage 6:Knowledge Node Recorder (Mastery Vault)
# =====================================================================
def refresh_vault(session_mgr):
    if not session_mgr or not session_mgr.mastered_nodes: return gr.update(choices=["No unlocked nodes yet"], value="No unlocked nodes yet"), "", ""
    choices = [f"{k} | {v['title']}" for k, v in session_mgr.mastered_nodes.items()]
    return gr.update(choices=choices, value=choices[0]), load_vault_node(choices[0], session_mgr)[0], load_vault_node(choices[0], session_mgr)[1]

def load_vault_node(selected, session_mgr):
    if not selected or "No unlocked" in selected: return "", ""
    node = session_mgr.mastered_nodes.get(selected.split(" | ")[0], {})
    content = f"### {node.get('title')}\n\n**Core Links**: {', '.join(node.get('linked_bases', []))}\n\n---\n{node.get('rich_text')}"
    return format_math_for_gradio(content), node.get("notes", "")

def save_note(selected, note_text, session_mgr):
    if selected and "No unlocked" not in selected: session_mgr.mastered_nodes[selected.split(" | ")[0]]["notes"] = note_text
    return "✅ Insights permanently synced to the Cognitive Graph!"



In [51]:
# =====================================================================
# STAGE 7: UI Assembly & Application Launch (Production Stable)
# =====================================================================

import torch
import gradio as gr
import re
import json

device = "cuda" if torch.cuda.is_available() else "cpu"

# =========================================================
# [1] CSS Theme Configurations (High-Contrast & Layout Fixes)
# =========================================================
css = """
body, .gradio-container {
    background-color: #f8fafc !important;
    font-family: 'Inter', -apple-system, sans-serif !important;
    color: #0f172a !important; 
}

.prose, .prose p, .prose div, .prose li {
    color: #0f172a !important;
    font-size: 17px !important; 
    line-height: 1.75 !important;
}

.prose span { color: inherit; }

.katex {
    font-size: 1.15em !important;
    color: #000000 !important; 
}

h1, h2, h3 { color: #0f172a !important; font-weight: 700 !important; letter-spacing: -0.02em !important; }

.prose span.example-badge {
    display: inline-block;
    background-color: #4f46e5 !important;
    color: #ffffff !important;
    padding: 2px 10px;
    border-radius: 6px;
    font-weight: 600;
    font-size: 0.85em;
    margin-right: 8px;
    letter-spacing: 0.05em;
    box-shadow: 0 2px 6px rgba(79, 70, 229, 0.3);
    vertical-align: middle;
    transform: translateY(-2px);
}

.arena-container {
    border: 1px solid #e2e8f0 !important;
    padding: 32px !important;
    background: #ffffff !important;
    border-radius: 12px !important;
    box-shadow: 0 10px 25px -5px rgba(0, 0, 0, 0.05) !important;
}

.tutor-container, .mastery-vault {
    border: 1px solid #e2e8f0 !important;
    background: #f8fafc !important;
    border-radius: 12px !important;
    padding: 24px !important;
}

/* Input field normalizations: Enforce high contrast */
.gradio-container textarea, 
.gradio-container input {
    background-color: #ffffff !important;
    color: #0f172a !important;
    border: 1px solid #cbd5e1 !important;
}

/* Chat bubble normalizations */
.message {
    background-color: #ffffff !important;
    border: 1px solid #e2e8f0 !important;
    color: #0f172a !important;
}

/* Distinct background for user messages */
.message.user {
    background-color: #f1f5f9 !important; 
}

/* Prevent Gradio internal paragraph styling overrides */
.message .prose p, .message .prose span {
    color: #0f172a !important;
}

.dark .prose { color: inherit !important; }
"""

# =========================================================
# [2] Text & Formula Pre-processing
# =========================================================
def format_math_for_gradio(text):
    if not text: 
        return ""
    text = "\n".join([line.lstrip() for line in text.split("\n")])
    text = re.sub(r"\\vspace\{.*?\}", "", text)
    text = re.sub(r"\\hspace\{.*?\}", "", text)
    text = re.sub(r"\\noindent\s*", "", text)
    text = text.replace("\\begin{itemize}", "").replace("\\end{itemize}", "")
    text = re.sub(r"\\item\s+", "• ", text)
    text = re.sub(r"\\textbf\{(.*?)\}", r"**\1**", text)
    # Parses Chinese example tags (e.g., 【例1】) into UI badges
    text = re.sub(r"【(例\s*[^】]+)】", r'<span class="example-badge">\1</span>', text)
    return text

def split_q_and_a(rich_text):
    parts = re.split(r'(\\textbf\{[解析证明]+[：\:]?\})', rich_text, maxsplit=1)
    if len(parts) >= 3: 
        return parts[0], parts[1] + parts[2]
    return rich_text, "*No official solution provided in the database.*"

# =========================================================
# [3] Responsive UI Interactions (Yield-based Streaming)
# =========================================================
def load_next_problem(session_mgr):
    if not session_mgr:
        return "System Offline.", "", gr.update(visible=True), gr.update(visible=False), gr.update(visible=False), session_mgr
        
    res = session_mgr.recommend_next_content()
    if res.get("status") == "empty":
        return "### Graph Mastered.\nYou have conquered all available nodes in the matrix.", "", gr.update(visible=False), gr.update(visible=False), gr.update(visible=False), session_mgr
    
    session_mgr._current_quest_id = res["recommended_id"]
    q_text, a_text = split_q_and_a(res["rich_text"])
    session_mgr._current_q_text = q_text
    session_mgr._current_a_text = a_text
    session_mgr._current_tools = res["linked_bases"]
    
    return format_math_for_gradio(q_text), "", gr.update(visible=True, interactive=True), gr.update(visible=False), gr.update(visible=False), session_mgr


def submit_answer_and_evaluate(user_ans, chat_history, session_manager):
    if not user_ans: 
        yield gr.update(), chat_history, gr.update(), gr.update(), gr.update(), session_manager
        return

    q_id = getattr(session_manager, "_current_quest_id", None)
    q_text = getattr(session_manager, "_current_q_text", "Unknown")
    a_text = getattr(session_manager, "_current_a_text", "None")

    chat_history.append({"role": "user", "content": f"📝 Submitted Solution:\n{user_ans}"})
    chat_history.append({"role": "assistant", "content": "⏳ *Gemma-4 is evaluating your logic. Please wait (approx. 10-20s)...*"})
    yield gr.update(interactive=False), chat_history, gr.update(), gr.update(), gr.update(), session_manager

    prompt = f"Evaluate student solution.\nProblem: {q_text}\nStudent Answer: {user_ans}\nReturn JSON only: {{\"score\": 0-100, \"feedback\": \"...\", \"status\": \"CORRECT|PARTIAL|INCORRECT\"}}"
    data = gemma_generate_json(prompt)
    
    if isinstance(data, str):
        try: 
            data = json.loads(data)
        except: 
            data = {"score": 30, "feedback": f"System Error: Model output format is invalid. Raw output: {data}", "status": "INCORRECT"}
            
    if not isinstance(data, dict):
        data = {"score": 30, "feedback": "System Error: Cognitive engine failed to generate a response. Please resubmit.", "status": "INCORRECT"}

    score = float(data.get("score", 30)) / 100.0
    feedback = data.get("feedback", "No feedback provided.")
    status = data.get("status", "INCORRECT")

    session_manager.backpropagate_score(q_id, score)
    prefix = "🟢 CORRECT" if status == "CORRECT" else ("🟡 PARTIAL" if status == "PARTIAL" else "🔴 INCORRECT")
    chat_history[-1] = {"role": "assistant", "content": f"{prefix} ({int(score*100)})\n\n{feedback}"}

    yield gr.update(visible=False), chat_history, gr.update(visible=True), format_math_for_gradio(a_text), gr.update(visible=True), session_manager


def chat_with_tutor(user_msg, chat_history, session_manager):
    if not user_msg: 
        yield "", chat_history
        return
        
    q_text = getattr(session_manager, "_current_q_text", "Unknown")
    chat_history.append({"role": "user", "content": user_msg})
    chat_history.append({"role": "assistant", "content": "⏳ *Gemma-4 is formulating heuristic hints...*"})
    yield "", chat_history
    
    prompt = f"You are a top-tier math coach.\nCurrent Problem:\n{q_text}\n\nStudent Question:\n{user_msg}\n\nGive a heuristic hint. Do NOT reveal final answer."
    reply = gemma_generate_text(prompt) 
    
    chat_history[-1] = {"role": "assistant", "content": format_math_for_gradio(reply)}
    yield "", chat_history


def request_ai_hint(chat_history, session_manager):
    q_text = getattr(session_manager, "_current_q_text", "Unknown")
    weak_nodes = session_manager.diagnose_weak_bases(session_manager._current_quest_id)
    weak_text = "\n".join([f"- {x['name']} (mastery={x['mastery']:.2f})" for x in weak_nodes])

    chat_history.append({"role": "user", "content": "💡 Request scaffolded hints"})
    chat_history.append({"role": "assistant", "content": "⏳ *Scanning cognitive topology to customize your hints...*"})
    yield chat_history
    
    prompt = f"You are an elite mathematics tutor.\nProblem:\n{q_text}\nWeak Concepts:\n{weak_text}\n\nIdentify bottleneck and generate 3 scaffolded micro-problems. Do NOT reveal final answer."
    reply = gemma_generate_text(prompt)
    
    chat_history[-1] = {"role": "assistant", "content": format_math_for_gradio(reply)}
    yield chat_history


# =========================================================
# [4] Gradio Layout Construction
# =========================================================
with gr.Blocks(css=css) as demo:
    
    session_state = gr.State(engine if 'engine' in globals() else None)

    gr.Markdown("# NEXUS ENGINE 3.0 - STABLE RENDER VERSION")

    with gr.Row():
        with gr.Column(scale=5, elem_classes="arena-container"):
            gr.Markdown("### Problem Arena")
            
            problem_display = gr.Markdown(
                value="⏳ Booting system...",
                latex_delimiters=[
                    {"left": "$$", "right": "$$", "display": True},
                    {"left": "$", "right": "$", "display": False},
                    {"left": "\\[", "right": "\\]", "display": True},
                    {"left": "\\(", "right": "\\)", "display": False}
                ]
            )
            
            user_answer = gr.Textbox(label="Your Solution", lines=8, interactive=True)
            
            with gr.Row():
                submit_btn = gr.Button("Submit Analysis")
                next_btn = gr.Button("Next Challenge")
            
            eval_result = gr.Markdown(visible=False)
            
            with gr.Column(visible=False) as official_sol_container:
                gr.Markdown("### Official Resolution")
                official_solution_text = gr.Markdown(
                    latex_delimiters=[
                        {"left": "$$", "right": "$$", "display": True},
                        {"left": "$", "right": "$", "display": False}
                    ]
                )

        with gr.Column(scale=4):
            with gr.Tabs():
                with gr.Tab("Resident Tutor", elem_classes="tutor-container"):
                    chatbot = gr.Chatbot(label="Tutor", height=480, show_label=False)
                    with gr.Row():
                        chat_input = gr.Textbox(show_label=False, placeholder="Ask something...", scale=4, interactive=True)
                        hint_btn = gr.Button("Hint", scale=1)

    # --- Event Binding ---
    demo.load(
        fn=load_next_problem,
        inputs=[session_state],
        outputs=[problem_display, user_answer, submit_btn, eval_result, official_sol_container, session_state]
    )
    
    next_btn.click(
        fn=load_next_problem,
        inputs=[session_state],
        outputs=[problem_display, user_answer, submit_btn, eval_result, official_sol_container, session_state]
    )
    
    submit_btn.click(
        fn=submit_answer_and_evaluate,
        inputs=[user_answer, chatbot, session_state],
        outputs=[submit_btn, chatbot, eval_result, official_solution_text, official_sol_container, session_state]
    )
    
    chat_input.submit(
        fn=chat_with_tutor,
        inputs=[chat_input, chatbot, session_state],
        outputs=[chat_input, chatbot]
    )
    
    hint_btn.click(
        fn=request_ai_hint,
        inputs=[chatbot, session_state],
        outputs=[chatbot]
    )

demo.launch(share=True)

/tmp/ipykernel_57/1584202675.py:199: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=css) as demo:


* Running on local URL:  http://127.0.0.1:7877
* Running on public URL: https://604e10edf07be102dd.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
